# PA 766 | 1. Convert PDFs to text

**Goal:** turn the class PDFs into `.txt` files and check whether their text is readable.

Open this notebook in **Google Colab** using **File > Upload notebook**. Use a standard CPU runtime. Run the cells from top to bottom using the play buttons. No API key is needed.

Download the example PDFs supplied by your instructor from the course's **Data / PDF to Text** folder. Start with `D00000-ElectricVehicleScenarioAnalysisWorkshopSeries.pdf`; you can select all four PDFs when ready.

You will download one ZIP containing the text files. Keep it for Notebook 2.

## 1. Install and import the tools

PyMuPDF reads the existing text inside a PDF. These examples contain text layers. A scanned image needs optical character recognition (OCR) first; this notebook does not perform OCR.

In [ ]:
%pip -q install pymupdf==1.28.2

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import pymupdf
import pandas as pd
from google.colab import files
from IPython.display import display

print("PyMuPDF version:", pymupdf.VersionBind)

## 2. Upload the PDFs

Click **Choose Files** and select one or more of the example PDFs from your computer. Uploading places a copy in the temporary Colab session; it does not edit your original files.

In [ ]:
uploaded = files.upload()
pdf_inputs = {name: data for name, data in uploaded.items()
              if name.lower().endswith(".pdf")}
if not pdf_inputs:
    raise ValueError("No PDFs selected. Run this cell again and choose PDF files.")
print(f"Ready to convert {len(pdf_inputs)} PDF(s).")

## 3. Extract and save the text

For each PDF, we read one page at a time. We keep text blocks separated by blank lines and replace line breaks *within* a block with spaces. A PDF block is a layout region, so it does not always equal a paragraph.

The invisible page-break character `\f` separates pages in each `.txt` file. Notebook 2 uses it to recover the source page number. We retain headings, footers, and table text so you can inspect what the extractor produced.

The table flags pages with fewer than 80 extracted characters. A flagged page may be a cover, divider, chart, or scanned page. This is a prompt to inspect the PDF, not an automatic diagnosis. Other pages can also have extraction errors.

In [ ]:
output_dir = Path("pa766_text_files")
output_dir.mkdir(exist_ok=True)
text_paths = []
summary = []

for filename, pdf_bytes in sorted(pdf_inputs.items()):
    page_texts = []
    low_text_pages = []
    with pymupdf.open(stream=pdf_bytes, filetype="pdf") as pdf:
        for page_number, page in enumerate(pdf, start=1):
            blocks = page.get_text("blocks", sort=True)
            paragraphs = [" ".join(block[4].split())
                          for block in blocks if block[6] == 0 and block[4].strip()]
            page_text = "\n\n".join(paragraphs)
            page_texts.append(page_text)
            if len(page_text.strip()) < 80:
                low_text_pages.append(page_number)

    if not any(text.strip() for text in page_texts):
        raise ValueError(f"{filename}: no text found. Ask your instructor about OCR.")

    text_path = output_dir / (Path(filename).stem + ".txt")
    text_path.write_text("\f".join(page_texts), encoding="utf-8")
    text_paths.append(text_path)
    summary.append({"source_pdf": filename, "pages": len(page_texts),
                    "words": sum(len(t.split()) for t in page_texts),
                    "pages_to_check": ", ".join(map(str, low_text_pages)) or "None flagged"})

display(pd.DataFrame(summary))

## 4. Check one page against the original

The cell shows PDF page 6 of the first uploaded document (or its last page if shorter). Change the file index or page number to inspect another page. Page numbers count from the first page of the PDF, including its cover.

Check whether sentences are complete, columns read in a sensible order, and headings or tables remain understandable. `sort=True` orders blocks by position; it cannot guarantee correct reading order for every layout. Charts and images are not transcribed. If important text is missing or scrambled, return to the PDF and ask your instructor before annotating it.

In [ ]:
FILE_INDEX = 0
PAGE_NUMBER = 6

chosen_path = text_paths[FILE_INDEX]
pages = chosen_path.read_text(encoding="utf-8").split("\f")
page_number = min(PAGE_NUMBER, len(pages))
if page_number < 1:
    raise ValueError("Choose a page number of at least 1.")
print(f"{chosen_path.name} | PDF page {page_number} of {len(pages)}\n")
print(pages[page_number - 1])

## 5. Download the text files

This ZIP contains only text files created in this run. Download it before leaving Colab, then open **PA766_02_Text_to_Annotation_CSV.ipynb**. In that notebook you can upload the ZIP directly, or unzip it on your computer and upload individual `.txt` files.

In [ ]:
zip_path = Path("PA766_text_files.zip")
with ZipFile(zip_path, "w") as archive:
    for text_path in text_paths:
        archive.write(text_path, arcname=text_path.name)
files.download(str(zip_path))

### Before moving on

- Identify one page where extraction worked well and one feature that needs inspection.
- Explain why successfully creating a text file does not guarantee that all PDF content was captured.

Technical reference: [PyMuPDF text extraction documentation](https://pymupdf.readthedocs.io/en/latest/recipes-text.html).